# bias-correction-divide — ex3: inflation factor 1/(1-beta**t) decays monotonically to 1

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `bias-correction-divide`. Running the final beacon cell reports progress against the `Optimizer: Adam bias-correction divide` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Optimizer: Adam bias-correction divide` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`bias-correction-divide`** (exercise 3). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "bias-correction-divide"
DD_SUBTOPIC = "Optimizer: Adam bias-correction divide"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Bias-correction's inflation factor `1/(1 - beta**t)` over time

Ex1 computed `m_hat = m / (1 - beta**t)` for one step. Ex2 traced the raw `m` trajectory under a constant gradient. The deepening move isolates the CORRECTION FACTOR itself — the multiplier `1/(1 - beta**t)` — and analyzes how it decays from large (at t=1) toward 1 (as t→∞).

```python
# At t=1 with beta=0.9:  1 / (1 - 0.9**1) = 1 / 0.1  = 10.0
# At t=10:               1 / (1 - 0.9**10) ≈ 1 / 0.651 ≈ 1.535
# At t=100:              1 / (1 - 0.9**100) ≈ 1.000027
```

**Monotone decreasing.** `beta**t` is monotonically decreasing in `t` (for `0 < beta < 1`), so `1 - beta**t` is monotonically INCREASING, and its reciprocal — the inflation factor — is monotonically DECREASING. Each subsequent step gets a smaller boost.

**Lower bound = 1.** `beta**t → 0` as `t → ∞`, so the factor approaches `1/(1 - 0) = 1`. Bias correction becomes a no-op at large step counts — the EMA has 'warmed up' and no longer needs help.

**Practical takeaway.** Bias correction matters MOST during the first ~`1/(1-beta)` steps (≈10 steps for beta=0.9, ≈1000 for beta=0.999). Past that horizon, the factor is essentially 1 and the divide is numerical-noise.

### Exercise 3 — inflation factor 1/(1-beta**t) decays monotonically to 1

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyze the bias-correction factor `1/(1 - beta**t)` across step indices t=1..n, demonstrating it is monotonically decreasing in t and asymptotes to 1, with rate controlled by beta.
> Keywords: bias-correction, inflation-factor, decay, warmup
> ```

**KCs targeted:** `inflation-factor-equals-reciprocal-one-minus-beta-to-t`, `inflation-factor-monotone-decreasing-to-one`

Implement `ex3_inflation_factor_trajectory(beta, n_steps)`. Compute the bias-correction inflation factor `1/(1 - beta**t)` for `t = 1, 2, ..., n_steps` and return it as a list of Python floats.

Inputs:
- `beta`: float in `(0, 1)` — the EMA decay.
- `n_steps`: int, `n_steps >= 1`.

Return: `list[float]` of length `n_steps`, where the `i`-th entry (0-indexed) is `1 / (1 - beta**(i+1))`. The 1-based step index matches the Adam convention.

Implementation: pure Python `**` and arithmetic — no tensor ops needed. Each entry is a regular Python float.

In [ ]:
def ex3_inflation_factor_trajectory(beta: float, n_steps: int) -> list:
    """Bias-correction inflation factors 1/(1-beta**t) for t=1..n_steps."""
    raise NotImplementedError()


def _test_ex3():
    # === Basic shape ===
    out = ex3_inflation_factor_trajectory(0.9, 5)
    assert isinstance(out, list), f'must return list, got {type(out)}'
    assert len(out) == 5, f'length must be n_steps=5, got {len(out)}'
    assert all(isinstance(x, float) for x in out), 'entries must be floats'

    # === Closed-form values at beta=0.9 ===
    # t=1:  1/(1-0.9)        = 10.0
    # t=2:  1/(1-0.81)       ≈ 5.2632
    # t=10: 1/(1-0.9**10)    ≈ 1.5354
    assert abs(out[0] - 10.0) < 1e-9, f'beta=0.9 t=1 should be 10.0; got {out[0]}'
    assert abs(out[1] - 1/(1-0.81)) < 1e-9, f'beta=0.9 t=2 mismatch; got {out[1]}'

    # === Monotone decreasing — the headline property ===
    for beta in [0.5, 0.9, 0.99, 0.999]:
        traj = ex3_inflation_factor_trajectory(beta, 50)
        for i in range(len(traj) - 1):
            assert traj[i] >= traj[i+1], (
                f'inflation factor must be monotone decreasing for beta={beta}; '
                f'failed at i={i}: {traj[i]} < {traj[i+1]}'
            )

    # === Asymptotes to 1 ===
    # For beta=0.9, by t=200 we should be essentially at 1.0.
    long_run = ex3_inflation_factor_trajectory(0.9, 300)
    assert abs(long_run[-1] - 1.0) < 1e-6, (
        f'inflation factor at t=300, beta=0.9 should be ~1.0; got {long_run[-1]}'
    )
    # Smaller beta converges FASTER to 1.
    fast = ex3_inflation_factor_trajectory(0.5, 20)
    assert abs(fast[-1] - 1.0) < 1e-5, f'beta=0.5 t=20 should be ~1.0; got {fast[-1]}'

    # === Larger beta → larger factor at fixed t ===
    # At t=10: beta=0.999 gives a much larger factor than beta=0.9.
    t10_slow = ex3_inflation_factor_trajectory(0.999, 10)[-1]
    t10_fast = ex3_inflation_factor_trajectory(0.9, 10)[-1]
    assert t10_slow > t10_fast, (
        f'slower decay (beta=0.999) needs larger correction at t=10 than beta=0.9; '
        f'got {t10_slow} vs {t10_fast}'
    )

    # === Always > 1.0 (factor INFLATES, never deflates) ===
    for beta in [0.5, 0.9, 0.99, 0.999]:
        traj = ex3_inflation_factor_trajectory(beta, 30)
        for i, v in enumerate(traj):
            assert v > 1.0 or abs(v - 1.0) < 1e-9, (
                f'factor must be >= 1 for beta={beta} t={i+1}; got {v}'
            )

    # === n_steps=1 edge case ===
    single = ex3_inflation_factor_trajectory(0.9, 1)
    assert len(single) == 1
    assert abs(single[0] - 10.0) < 1e-9, f'single-step at beta=0.9 should be 10.0; got {single[0]}'

    # === Effective horizon rule of thumb: at t ~ 1/(1-beta), factor ≈ ~1.58 ===
    # For beta=0.9, horizon is 10. inflation_factor[10-1] (1-indexed t=10) ≈ 1.535
    h_traj = ex3_inflation_factor_trajectory(0.9, 10)
    assert 1.4 < h_traj[-1] < 1.7, (
        f'at t=1/(1-beta)=10, factor should be in [1.4, 1.7]; got {h_traj[-1]}'
    )
    _dd_passed.add('ex3')
    print("ex3 ✓")

_test_ex3()

<details><summary>Solution</summary>

```python
def ex3_inflation_factor_trajectory(beta, n_steps):
    return [1.0 / (1.0 - beta ** t) for t in range(1, n_steps + 1)]
```

**The 1-based loop is mandatory.** Adam's bias correction is `1 - beta**t` with `t` starting at 1 (not 0). Using `range(0, n)` would put a `1/(1 - 1) = 1/0` divide-by-zero at the first step.

**Effective horizon = `1/(1-beta)`.** Substituting `t = 1/(1-beta)` gives `beta**t ≈ exp(-1) ≈ 0.368` (for beta near 1), so the factor is `1/(1 - 0.368) ≈ 1.58`. Past this horizon, bias correction has done most of its work and the factor approaches 1 quickly.

**The list-comp is `O(n)`.** For Adam in real training you'd never materialize this list — you just compute the current step's factor on the fly. The materialized trajectory exists for plotting / analysis only.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex3',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()